# MODULE 6 : <font color='cyan'>**PERSISTANCE INDUSTRIELLE & MODÉLISATION RELATIONNELLE** (SQLite3)</font>

**🎯 Objectif :** *Passer des fichiers plats (JSON) à une base de données relationnelle structurée.*

* **Modélisation SQL :** Tables, types de données (`TEXT`, `INTEGER`, `REAL`).
* **Clé Primaire (`PRIMARY KEY`) :** Unicité des enregistrements et auto-incrémentation.
* **Clés Étrangères (`FOREIGN KEY`) :** Mise en place de relations 1:N entre les tables.
* **Intégrité des données :** Contraintes `NOT NULL`, `UNIQUE` et `CHECK`.
* **Requêtes CRUD :** Syntaxes `INSERT INTO`, `SELECT` (+ `WHERE`, `LIKE`, `ORDER BY`), `UPDATE` et `DELETE`.
* **Connecteur `sqlite3` :** Connexion, Curseur (`execute`, `fetchall`), `commit()` et `rollback()`.


# <font color='yellow'>I. **GENERALITES**</font>

Jusqu'ici, pour sauvegarder des données, nous utilisions des fichiers texte ou JSON.

C'est une très bonne première approche, mais lorsqu'un programme grandit, les fichiers JSON montrent rapidement leurs limites :

* Pour modifier un seul utilisateur dans un JSON de 10 000 lignes, il faut recharger **tout le fichier** en mémoire, le modifier, puis le réécrire entièrement sur le disque.
* Si deux programmes essaient d'écrire dans le même fichier JSON en même temps, le fichier risque d'être corrompu.
* On ne peut pas facilement garantir qu'un champ sera toujours unique (comme un e-mail ou un numéro de téléphone) sans réécrire beaucoup de code Python.

C'est là qu'interviennent les **Bases de Données Relationnelles**.

## 1. **Qu'est-ce qu'une Base de Données et le langage SQL ?**

Une **Base de Données Relationnelle (SGBDR)** est un logiciel spécialisé dans le stockage, la structuration et la recherche rapide de données.

* **Une Table :** Dans une base de données, les données sont organisées en **tables**. Une table ressemble à un tableau structuré avec des **colonnes** (les champs) et des **lignes** (les enregistrements).
* **Le langage SQL (*Structured Query Language*) :** C'est le langage universel utilisé pour communiquer avec la base de données. Que ce soit en Python, Java ou C#, on envoie des instructions SQL à la base de données pour lui dire quoi créer, insérer, chercher ou modifier.



## 2. **SQLite : La Base de Données Intégrée à Python**

Pour démarrer, nous utilisons **SQLite**.

C'est un moteur de base de données extrêmement puissant et léger :

* Il ne nécessite **aucun serveur à installer**.
* Toute la base de données est stockée dans **un seul fichier sur ton disque** (par exemple `app.db`).
* Python intègre déjà le module natif `sqlite3`, il n'y a donc rien à installer avec `pip` !


## 3. **Les Types de Données en SQL (SQLite)**

Avant de créer une table, il faut définir le type de chaque colonne. Voici les types principaux en SQLite :

|  Type SQL      | Equivalent Python | Description / Usage                               |
| -------------- | ----------------- | ------------------------------------------------- |
| **`INTEGER`**  | `int`             | Nombres entiers (ex: âge, identifiant, quantité). |
| **`REAL`**     | `float`           | Nombres à virgule (ex: prix, note, température).  |
| **`TEXT`**     | `str`             | Chaînes de caractères / texte (ex: nom, e-mail).  |
| **`NULL`**     | `None`            | Représente une valeur absente ou inconnue.        |


## 4. **Créer sa première Table avec `CREATE TABLE`**

Voici la syntaxe SQL pour créer une table nommée `etudiants` :

```sql
CREATE TABLE etudiants (
    id INTEGER,
    nom TEXT,
    age INTEGER,
    moyenne REAL
);

```

### **Explication** :

* `CREATE TABLE etudiants` : Demande à la base de données de créer la table `etudiants`.
* Entre parenthèses, on liste les colonnes suivies de leur type, séparées par des virgules.


## 5. **Exécuter du SQL en Python avec `sqlite3`**

Pour manipuler une base de données depuis un script Python, on suit toujours ces 4 étapes :

```
Connexion (connect) ──> Curseur (cursor) ──> Exécution (execute) ──> Validation (commit)

```

1. **`sqlite3.connect("ma_base.db")`** : Ouvre le fichier de base de données (et le crée s'il n'existe pas).
2. **`connexion.cursor()`** : Crée un **curseur** (l'outil qui envoie les commandes SQL à la base).
3. **`curseur.execute(...)`** : Envoie l'instruction SQL.
4. **`connexion.commit()`** : Valide et enregistre définitivement les modifications sur le disque.
5. **`connexion.close()`** : Ferme la connexion proprement.


### **Code d'exemple complet** :

```python
import sqlite3

# 1. Connexion à la base de données (crée le fichier 'ecole.db')
connexion = sqlite3.connect("ecole.db")

# 2. Création du curseur
curseur = connexion.cursor()

# 3. Création de la table 'etudiants' si elle n'existe pas (IF NOT EXISTS)
curseur.execute("""
CREATE TABLE IF NOT EXISTS etudiants (
    id INTEGER,
    nom TEXT,
    age INTEGER,
    moyenne REAL
)
""")

# 4. Validation des changements
connexion.commit()

# 5. Fermeture de la connexion
connexion.close()

print("La base de données et la table ont été créées avec succès !")

```

### Exercice 6.1 : **À toi de jouer !** 

**Contexte :** Tu développes un système pour gérer une bibliothèque de livres.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à une base de données nommée `bibliotheque.db`.
2. Crée une table nommée `livres` contenant les 4 colonnes suivantes :
   * `id` de type entier (`INTEGER`)
   * `titre` de type texte (`TEXT`)
   * `auteur` de type texte (`TEXT`)
   * `prix` de type nombre à virgule (`REAL`)

3. Valide les changements (`commit`) et ferme la connexion.

---

## <font color='yellow'>II. **NOTIONS DE CLE PRIMAIRE**</font>

Dans l'exercice précédent, nous avons créé une table avec une colonne `id INTEGER`. Cependant, rien n'empêchait d'avoir deux étudiants ou deux livres avec exactement le même identifiant.

Pour garantir l'unicité absolue de chaque ligne, on utilise une **Clé Primaire** (`PRIMARY KEY`).


## 1. **Qu'est-ce qu'une Clé Primaire (`PRIMARY KEY`) ?**

La **clé primaire** est la colonne (ou l'ensemble de colonnes) qui identifie **de façon unique** chaque enregistrement dans une table.

* **Unique :** Deux lignes ne peuvent pas avoir la même valeur de clé primaire.
* **Obligatoire :** Elle ne peut jamais être vide (`NULL`).
* **Auto-incrémentée :** En SQLite, si une colonne est définie comme `INTEGER PRIMARY KEY AUTOINCREMENT`, la base de données génère automatiquement un nouveau numéro (1, puis 2, puis 3...) à chaque fois que tu ajoutes une ligne !


##  2. **Syntaxe SQL avec `PRIMARY KEY`**

Reprenons la création d'une table `utilisateurs` en ajoutant une clé primaire auto-incrémentée :

```sql
CREATE TABLE utilisateurs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT,
    email TEXT
);

```

### **Explication** :

* `id INTEGER PRIMARY KEY AUTOINCREMENT` : La base s'occupe seule de donner un `id` unique à chaque nouvel utilisateur. Tu n'as plus besoin d'inventer ou de calculer les identifiants !


## 3. **Exemple en Python avec `sqlite3`**

```python
import sqlite3

connexion = sqlite3.connect("application.db")
curseur = connexion.cursor()

# Création de la table avec une clé primaire auto-incrémentée
curseur.execute("""
CREATE TABLE IF NOT EXISTS utilisateurs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT,
    email TEXT
)
""")

connexion.commit()
connexion.close()

print("Table 'utilisateurs' créée avec une clé primaire !")

```


### Exercice 6.2 : **À toi de jouer** ! 

**Contexte :** Tu gères le répertoire d'un magasin de jeux vidéo.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à la base de données `magasin.db`.
2. Crée une table nommée `jeux` avec les colonnes suivantes :
   * `id` : entier, clé primaire auto-incrémentée (`INTEGER PRIMARY KEY AUTOINCREMENT`).
   * `titre` : texte (`TEXT`).
   * `console` : texte (`TEXT`).
   * `prix` : nombre à virgule (`REAL`).

3. Valide les modifications et ferme la connexion.


---

# <font color='yellow'>III. **LES REQUETES**</font>

Par définition, une **requête** est une commande adressée à la base de données pour lire, ajouter, modifier ou supprimer des données.

Maintenant que la structure de notre table est en place avec une clé primaire, nous allons y **insérer des données** grâce à l'instruction SQL **`INSERT INTO`**.

## a. <font color='yellow'>**La requête d'insertion des données (`INSERT INTO`)**</font>


##  1. **La Syntaxe SQL `INSERT INTO`**

Pour ajouter une nouvelle ligne dans une table, la syntaxe SQL de base est la suivante :

```sql
INSERT INTO nom_de_la_table (colonne1, colonne2, colonne3) VALUES ('valeur1', 'valeur2', 'valeur3');

```


### <font color='rgba(211, 50, 10, 0.7)'>**Remarques importantes**</font> :
   > * On liste les colonnes dans lesquelles on souhaite insérer des données.
   > * **Important :** Si la colonne `id` est configurée en `INTEGER PRIMARY KEY AUTOINCREMENT`, on **ne la mentionne pas** dans le `INSERT INTO`. La base de données va lui attribuer son numéro automatiquement !
   > * Les valeurs de type texte (`TEXT`) s'écrivent entre guillemets simples `'...'`.

## 2. **Insérer des données depuis Python (Sécurité & Bonnes Pratiques)**

En Python avec `sqlite3`, il existe deux manières d'exécuter un `INSERT INTO` :

### A. **La mauvaise méthode (Concaténation de chaînes - À ÉVITER)**

```python
# ❌ DANGEREUX : vulnérable aux injections SQL et pose des problèmes avec les guillemets/apostrophes
nom = "D'Arc"
curseur.execute(f"INSERT INTO utilisateurs (nom) VALUES ('{nom}')")  # Plante !

```

### B. **La bonne méthode (Les paramètres fantômes `?`)**

On utilise des **point d'interrogation `?**` comme emplacements réservés. Le module `sqlite3` s'occupe de sécuriser et de formater correctement les données.

```python
import sqlite3

connexion = sqlite3.connect("magasin.db")
curseur = connexion.cursor()

# 1. Insertion d'UNE SEULE ligne avec executemany/execute et des tuples
nom_jeu = "Super Mario Odyssey"
console = "Nintendo Switch"
prix = 49.99

curseur.execute(
    "INSERT INTO jeux (titre, console, prix) VALUES (?, ?, ?)",
    (nom_jeu, console, prix),
)

# 2. Validation OBLIGATOIRE des changements sur le disque
connexion.commit()
connexion.close()

print("Jeu ajouté avec succès !")

```

## 3. **Insérer plusieurs lignes d'un coup avec `executemany()`**

Si tu as une liste de données à insérer, au lieu de faire une boucle `for` avec `.execute()`, tu peux utiliser **`.executemany()`** qui est beaucoup plus rapide :

```python
nouveaux_jeux = [
    ("Zelda: Breath of the Wild", "Nintendo Switch", 59.99),
    ("God of War", "PlayStation 5", 69.99),
    ("Minecraft", "PC", 29.99),
]

# Insère toute la liste d'un seul coup
curseur.executemany(
    "INSERT INTO jeux (titre, console, prix) VALUES (?, ?, ?)", nouveaux_jeux
)

connexion.commit()

```

### **À toi de jouer** ! (Exercice 6.3)

**Contexte :** Tu alimentes la base de données de ton magasin de jeux vidéo.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à la base `magasin.db`.
2. S'assure que la table `jeux` existe (`id`, `titre`, `console`, `prix`).
3. Insère une liste d'au moins 3 jeux vidéo de ton choix en utilisant `.executemany()`.
4. Enregistre les modifications avec `.commit()` et ferme la connexion.


# b. <font color='yellow'>**La requête de lecture ou de selection des données (`SELECT`)**</font>

Une fois que tes données sont enregistrées dans la base, il faut pouvoir les **lire et les récupérer** dans ton programme Python. Pour cela, on utilise la commande SQL **`SELECT`**.


## 1. **La Syntaxe SQL `SELECT`**

Le mot-clé `SELECT` permet de spécifier quelles colonnes tu souhaites lire, et `FROM` indique la table ciblée.

```sql
-- Récupérer TOUTES les colonnes de tous les jeux
SELECT * FROM jeux;

-- Récupérer seulement les colonnes 'titre' et 'prix'
SELECT titre, prix FROM jeux;

```

> 💡 **Le symbole `*` (étoile) :** Il signifie *"toutes les colonnes"*.


## 2. **Filtrer et Trier les Résultats**

Pour affiner ta recherche, tu peux ajouter des clauses à ta requête SQL :

* **`WHERE` (Filtrer) :** Sélectionne uniquement les lignes qui respectent une condition.
* **`ORDER BY` (Trier) :** Trie les résultats selon une colonne (`ASC` pour croissant, `DESC` pour décroissant).
* **`LIKE` (Recherche textuelle) :** Cherche un mot ou un morceau de texte grâce au joker `%`.

```sql
-- Trouver tous les jeux de la console 'Nintendo Switch'
SELECT * FROM jeux WHERE console = 'Nintendo Switch';

-- Trouver les jeux qui coûtent moins de 50.0 € et les trier du moins cher au plus cher
SELECT * FROM jeux WHERE prix < 50.0 ORDER BY prix ASC;

-- Chercher tous les jeux dont le titre contient le mot 'Mario'
SELECT * FROM jeux WHERE titre LIKE '%Mario%';

```


## 3. **Récupérer les Résultats en Python (`fetchall` et `fetchone`)**

Une fois la requête `SELECT` exécutée avec `curseur.execute()`, la base de données met les résultats en attente. Tu les récupères avec l'une de ces deux méthodes :

1. **`curseur.fetchall()` :** Renvoie une **liste de tuples** contenant toutes les lignes trouvées.
2. **`curseur.fetchone()` :** Renvoie **un seul tuple** (la première ligne trouvée), ou `None` si rien n'a été trouvé.

```python
import sqlite3

connexion = sqlite3.connect("magasin.db")
curseur = connexion.cursor()

# 1. Exécuter la requête de sélection
curseur.execute("SELECT id, titre, console, prix FROM jeux WHERE prix < 60.0")

# 2. Récupérer TOUTES les lignes sous forme de liste
tous_les_jeux = curseur.fetchall()

# 3. Parcourir les résultats avec une boucle for
for jeu in tous_les_jeux:
  # Chaque 'jeu' est un tuple : (id, titre, console, prix)
  id_jeu, titre, console, prix = jeu
  print(f"[{id_jeu}] {titre} ({console}) - {prix} €")

connexion.close()

```

> 💡 **Remarque :** Pour les requêtes de lecture (`SELECT`), pas besoin de faire un `connexion.commit()`, car on ne modifie rien sur le disque ! Rappelle-toi, la méthode `commit()` sert à valider les changements dans la base de données.


### **À toi de jouer** ! (Exercice 6.4)

**Contexte :** Tu veux afficher le catalogue de ton magasin de jeux vidéo.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à la base de données `magasin.db`.
2. Effectue une requête `SELECT` pour récupérer uniquement les jeux de la console `"Nintendo Switch"` triés par prix décroissant (`ORDER BY prix DESC`).
3. Récupère tous les résultats avec `.fetchall()`.
4. Affiche chaque jeu dans le terminal sous la forme : `Titre - Prix €`.

# c. **<font color='yellow'>La requête de modification des données (`UPDATE`)</font>**

L'instruction `UPDATE` permet de changer la valeur d'une ou plusieurs colonnes dans des lignes existantes.

## 1. **Syntaxe SQL** :

```sql
UPDATE nom_de_la_table 
SET colonne1 = nouvelle_valeur, colonne2 = nouvelle_valeur
WHERE condition;

```

> ⚠️ **ATTENTION OBLIGATOIRE : La clause `WHERE**`
> Si tu oublies la clause `WHERE`, la modification s'appliquera à **TOUTES LES LIGNES** de ta table ! C'est pourquoi on cible presque toujours la modification à l'aide de la **Clé Primaire (`id`)**.

## 2. **Exemple en Python** :

```python
import sqlite3

connexion = sqlite3.connect("magasin.db")
curseur = connexion.cursor()

# Promouvoir une réduction sur le jeu dont l'id est 1
nouveau_prix = 39.99
id_jeu = 1

curseur.execute(
    "UPDATE jeux SET prix = ? WHERE id = ?", (nouveau_prix, id_jeu)
)

# Comme on MODIFIE la base, le commit() est OBLIGATOIRE !
connexion.commit()
connexion.close()

print("Le prix du jeu a été mis à jour !")

```

# d. **<font color='yellow'>La requête de suppression des données (`DELETE`)</font>**

L'instruction `DELETE FROM` permet de retirer définitivement un ou plusieurs enregistrements de la table.

## **Syntaxe SQL** :

```sql
DELETE FROM nom_de_la_table 
WHERE condition;

```

> ⚠️ **ATTENTION DANGER :** Tout comme pour `UPDATE`, si tu écris `DELETE FROM jeux;` sans clause `WHERE`, **toute ta table sera vidée** ! Utilise toujours l'identifiant unique `id` pour cibler la ligne exacte à supprimer.

## **Exemple en Python** :

```python
import sqlite3

connexion = sqlite3.connect("magasin.db")
curseur = connexion.cursor()

# Supprimer le jeu dont l'id est 3
id_a_supprimer = 3

curseur.execute("DELETE FROM jeux WHERE id = ?", (id_a_supprimer,))

# On confirme la suppression
connexion.commit()
connexion.close()

print("Le jeu a été supprimé de la base de données !")

```

## **Récapitulatif du cycle CRUD**

| Action | Commande SQL | Requier `commit()` ? |
| --- | --- | --- |
| **C**reate (Créer) | `INSERT INTO` | **Oui** |
| **R**ead (Lire) | `SELECT` | Non |
| **U**pdate (Modifier) | `UPDATE` | **Oui** |
| **D**elete (Supprimer) | `DELETE FROM` | **Oui** |


### **À toi de jouer** ! (Exercice 6.5)

**Contexte :** Tu mets à jour l'inventaire de ton magasin de jeux vidéo.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à la base de données `magasin.db`.
2. Applique une promotion : met à jour le prix du jeu ayant l' `id = 2` pour le passer à **19.99 €**.
3. Supprime de la base tous les jeux dont le prix est strictement supérieur à **60.0 €** (`WHERE prix > 60.0`).
4. Valide les modifications avec `.commit()` et ferme la connexion.

---

# **<font color='yellow'>IV. **LES CONTRAINTES D'INTEGRITE**</font>**

Jusqu'à présent, nous avons créé des colonnes avec de simples types (`TEXT`, `INTEGER`, `REAL`). Mais sans règles supplémentaires, n'importe qui pourrait insérer :

* Un utilisateur sans nom.
* Deux utilisateurs avec exactement le même e-mail.
* Un jeu avec un prix négatif (ex: `-15 €`).

Les **contraintes d'intégrité** sont des règles définies à la création de la table pour forcer la base de données à rejeter automatiquement les données invalides.


## <font color='yellow'>**Les 3 Contraintes Essentielles**</font>

### 1. `NOT NULL` (Obligatoire)

Empêche d'insérer une ligne où cette colonne est vide (`NULL`).

* *__Exemple__ :* Un utilisateur doit obligatoirement avoir un nom.

### 2. `UNIQUE` (Sans doublon)

Garantit que deux lignes ne peuvent pas avoir la même valeur dans cette colonne.

* *__Exemple__ :* Deux comptes ne peuvent pas partager la même adresse e-mail ou le même pseudo.

### 3. `CHECK` (Condition de validation)

Vérifie qu'une condition logique est respectée avant d'accepter la donnée.

* *__Exemple__ :* Le prix d'un produit doit être supérieur à zéro (`prix > 0`), ou l'âge d'un joueur doit être supérieur ou égal à 13 (`age >= 13`).


## <font color='yellow'> **Syntaxe SQL avec Contraintes**</font>

Voici comment appliquer ces contraintes lors du `CREATE TABLE` :

```sql
CREATE TABLE joueurs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pseudo TEXT NOT NULL UNIQUE, 
    email TEXT NOT NULL UNIQUE, 
    age INTEGER NOT NULL CHECK (age >= 13),
    niveau INTEGER DEFAULT 1 CHECK (niveau >= 1)
);

```

### **Explication** :

* `pseudo TEXT NOT NULL UNIQUE` : Le pseudo ne peut pas être vide **ET** doit être unique.
* `age INTEGER NOT NULL CHECK (age >= 13)` : L'âge est obligatoire **ET** doit être au minimum 13.
* Si quelqu'un tente d'insérer un joueur de 10 ans ou avec un pseudo déjà existant, SQLite bloque l'opération et lève une erreur en Python !

---

## <font color='yellow'>**Tester les Contraintes en Python**</font>

Lorsqu'une contrainte est violée, Python lève une exception du type `sqlite3.IntegrityError`.

```python
import sqlite3

connexion = sqlite3.connect("jeux.db")
curseur = connexion.cursor()

# 1. Création de la table sécurisée
curseur.execute("""
CREATE TABLE IF NOT EXISTS produits (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT NOT NULL,
    prix REAL NOT NULL CHECK (prix > 0)
)
""")

# 2. Insertion valide (Fonctionne)
curseur.execute(
    "INSERT INTO produits (nom, prix) VALUES (?, ?)", ("Manette PS5", 69.99)
)
connexion.commit()

# 3. Tentative d'insertion INVALIDE (Prix négatif)
try:
  curseur.execute(
      "INSERT INTO produits (nom, prix) VALUES (?, ?)", ("Jeu d'occasion", -10.0)
  )
  connexion.commit()
except sqlite3.IntegrityError as e:
  print("❌ Erreur bloquée par la base de données :", e)

connexion.close()

```

### **À toi de jouer** ! (Exercice 6.6)

**Contexte :** Tu créés le système d'inscription pour un club d'e-sport.

**Énoncé :**
Écris un script Python qui :

1. Se connecte à la base `esport.db`.
2. Crée une table `membres` avec les règles suivantes :
   * `id` : clé primaire auto-incrémentée (`INTEGER PRIMARY KEY AUTOINCREMENT`).
   * `pseudo` : texte, obligatoire et unique (`TEXT NOT NULL UNIQUE`).
   * `points` : nombre entier, obligatoire et doit être supérieur ou égal à 0 (`INTEGER NOT NULL CHECK (points >= 0)`).

3. Teste l'insertion d'un membre valide, puis d'un membre avec un nombre de points négatif dans un bloc `try/except sqlite3.IntegrityError` pour vérifier que l'erreur est bien capturée.

## **Récapitulation**

Une **contrainte d'intégrité** est tout simplement une règle qu'on impose à la base de données lors de la création d'une table (`CREATE TABLE`).

Son rôle est de faire le "gendarme" : la base de données **refusera automatiquement** d'enregistrer toute donnée qui ne respecte pas ces règles en levant une erreur `sqlite3.IntegrityError`.

### **Synthèse des 4 Contraintes Principales**

| Contrainte SQL | Rôle | Exemple Concret | Code SQL |
| --- | --- | --- | --- |
| **`NOT NULL`** | Rend le champ **obligatoire** (interdit les valeurs vides). | Un étudiant doit obligatoirement avoir un `nom`. | `nom TEXT NOT NULL` |
| **`UNIQUE`** | Interdit les **doublons** sur cette colonne. | Deux utilisateurs ne peuvent pas avoir le même `email`. | `email TEXT UNIQUE` |
| **`CHECK (...)`** | Valide une **condition logique** sur la valeur. | Le prix d'un article doit être strictement supérieur à zéro. | `prix REAL CHECK (prix > 0)` |
| **`PRIMARY KEY`** | Combine **`NOT NULL`** + **`UNIQUE`** pour identifier de façon unique chaque ligne. | Donner un numéro unique `id` à chaque joueur. | `id INTEGER PRIMARY KEY AUTOINCREMENT` |


### **Pourquoi les utiliser ?**

1. **Sécurité maximale :** Même si ton code Python contient un petit bug, la base de données bloque les mauvaises données au dernier niveau.
2. **Propreté des données :** Tu es garanti de ne jamais te retrouver avec des e-mails en double, des prix négatifs ou des profils sans nom.
3. **Moins de code Python :** Tu n'as pas besoin d'écrire des dizaines de `if/else` en Python pour vérifier si la donnée est valide ; SQLite le fait à ta place !
